In [1]:
import pandas as pd
import numpy as np

In [2]:
TRAIN_PATH = "/kaggle/input/ember-feature-based-dataset/ember/train_ember_2018_final.parquet"
TEST_PATH  = "/kaggle/input/ember-feature-based-dataset/ember/test_ember_2018_final.parquet"

train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

print(train_df.shape, test_df.shape)

(599920, 2342) (199956, 2342)


In [3]:
# Separate classes
malware = train_df[train_df["Label"] == 1].sample(frac=1, random_state=42)
benign  = train_df[train_df["Label"] == 0].sample(frac=1, random_state=42)

# Client-2 uses 50% of total training data
client_size = int(0.5 * len(train_df))

# 80% malware, 20% benign
mal_count = int(0.8 * client_size)
ben_count = int(0.2 * client_size)

client2_df = pd.concat([
    malware.iloc[:mal_count],
    benign.iloc[:ben_count]
]).sample(frac=1, random_state=42)

print(client2_df["Label"].value_counts())

Label
1    239968
0     59992
Name: count, dtype: int64


In [4]:
X_train = client2_df.drop(columns=["Label"]).values
y_train = client2_df["Label"].values

X_test = test_df.drop(columns=["Label"]).values
y_test = test_df["Label"].values

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

mlp = Sequential([
    Dense(512, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

mlp.compile(
    optimizer=Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

mlp.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    verbose=1
)

2026-01-05 21:48:40.304910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767649720.466102      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767649720.521616      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767649720.912206      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767649720.912249      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767649720.912251      55 computation_placer.cc:177] computation placer alr

Epoch 1/10


I0000 00:00:1767649741.908668     147 service.cc:152] XLA service 0x7de180005f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767649741.908710     147 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767649742.219825     147 cuda_dnn.cc:529] Loaded cuDNN version 91002


 37/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7837 - loss: 0.5610

I0000 00:00:1767649743.775988     147 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


528/528 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.9064 - loss: 0.2514 - val_accuracy: 0.9510 - val_loss: 0.1248
Epoch 2/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9531 - loss: 0.1212 - val_accuracy: 0.9579 - val_loss: 0.1061
Epoch 3/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9611 - loss: 0.1005 - val_accuracy: 0.9605 - val_loss: 0.0999
Epoch 4/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9654 - loss: 0.0880 - val_accuracy: 0.9575 - val_loss: 0.1056
Epoch 5/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9682 - loss: 0.0806 - val_accuracy: 0.9629 - val_loss: 0.0975
Epoch 6/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9712 - loss: 0.0737 - val_accuracy: 0.9626 - val_loss: 0.0998
Epoch 7/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9727 - loss: 0.0699 - val_accuracy: 0.9651 - val_loss: 0.0952
Epoch 8/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9748 - loss: 0.0648 - val_accuracy: 0.9654 - val

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [8]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

mlp_probs = mlp.predict(X_test).ravel()
rf_probs  = rf.predict_proba(X_test)[:, 1]

ensemble_probs = (mlp_probs + rf_probs) / 2
ensemble_preds = (ensemble_probs > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, ensemble_preds))
print("F1:", f1_score(y_test, ensemble_preds))
print("ROC-AUC:", roc_auc_score(y_test, ensemble_probs))

6249/6249 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step
Accuracy: 0.91078037168177
F1: 0.9164653218707273
ROC-AUC: 0.9881658220765004


In [9]:
mlp.save("ember_client2_mlp.keras")

import joblib
joblib.dump(rf, "ember_client2_rf.pkl")

['ember_client2_rf.pkl']